# App Review Sentiment Classification — Activities 1–5

This notebook trains a TF-IDF + Logistic Regression pipeline, evaluates it, plots a confusion matrix, inspects confident mistakes, compares the original model with a class-balanced model, and flags new feedback.

**Before running:** place the provided app-review dataset (CSV) in the same folder as this notebook. In the setup cell, update the filename and the text/label column names to match your dataset. The prompt did not include the dataset file or the unseen feedback snippets, so those inputs are left as clearly marked settings rather than inventing results.

## 1. Import libraries and configure the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42

# EDIT THESE SETTINGS to match your provided CSV:
DATA_PATH = "app_reviews.csv"   # e.g. "googleplay_reviews.csv"
TEXT_COLUMN = "review"          # column containing review text
LABEL_COLUMN = "sentiment"       # column containing labels (e.g. negative/positive)

# Set this to the exact label used for negative reviews in your dataset.
NEGATIVE_LABEL = "negative"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

If the dataset uses different column names, change `TEXT_COLUMN` and `LABEL_COLUMN` above. If labels are encoded as `0` and `1`, set `NEGATIVE_LABEL` to whichever value represents negative reviews.

## 2. Prepare the data and make a held-out test split

In [ ]:
# Keep only rows with both text and label present
data = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna().copy()
data[TEXT_COLUMN] = data[TEXT_COLUMN].astype(str)
data[LABEL_COLUMN] = data[LABEL_COLUMN].astype(str)

X = data[TEXT_COLUMN]
y = data[LABEL_COLUMN]

print("Class labels:", sorted(y.unique()))
print("\nClass distribution:")
display(y.value_counts().rename_axis("label").to_frame("count"))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

## Activity 1 — Train a full sklearn Pipeline and report metrics

In [ ]:
# Pipeline combines TF-IDF text features with Logistic Regression.
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
weighted_precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
weighted_recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision (weighted): {weighted_precision:.4f}")
print(f"Recall (weighted):    {weighted_recall:.4f}")
print(f"F1-score (weighted):  {weighted_f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

## Activity 2 — Plot and interpret the confusion matrix

In [ ]:
labels = list(pipeline.named_steps["classifier"].classes_)
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix — App Review Sentiment")
plt.tight_layout()
plt.show()

print("Rows are true labels; columns are predicted labels.")
print("Correct predictions are on the diagonal. Off-diagonal cells are errors.")

**Interpretation (write after viewing the plotted matrix):** The diagonal cells show how many reviews were classified correctly for each class. Off-diagonal cells show which classes were confused with one another. The class with the largest number of false negatives (its row's off-diagonal counts) is the class the model misses most often; the class with the largest false-positive count (its column's off-diagonal counts) is most often incorrectly predicted. Use the actual matrix values above to name the class rather than assuming one in advance.

## Activity 3 — Find misclassified examples and the model's confidence

In [ ]:
# predict_proba columns follow the order in pipeline.classes_
probabilities = pipeline.predict_proba(X_test)
class_names = pipeline.named_steps["classifier"].classes_
predicted_class_index = np.argmax(probabilities, axis=1)

# Confidence means probability assigned to the class the model actually predicted.
predicted_probability = probabilities[np.arange(len(probabilities)), predicted_class_index]

misclassified = pd.DataFrame({
    "text": X_test.to_numpy(),
    "true_label": y_test.to_numpy(),
    "predicted_label": y_pred,
    "predicted_probability": predicted_probability
})

misclassified = (
    misclassified[misclassified["true_label"] != misclassified["predicted_label"]]
    .sort_values("predicted_probability", ascending=False)
    .reset_index(drop=True)
)

print("Number of misclassified test examples:", len(misclassified))
display(misclassified.head(5))

The displayed top five are the model's most confident mistakes, because `predicted_probability` is the probability assigned to its predicted class. If fewer than five examples were misclassified, the table will show all available errors.

## Activity 4 — Class distribution and class-balanced retraining

In [ ]:
print("Class distribution using value_counts():")
display(data[LABEL_COLUMN].value_counts().rename_axis("label").to_frame("count"))

# Logistic Regression supports class_weight='balanced'.
balanced_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)),
    ("classifier", LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced"
    ))
])

balanced_pipeline.fit(X_train, y_train)
balanced_pred = balanced_pipeline.predict(X_test)

if str(NEGATIVE_LABEL) not in set(map(str, y_test.unique())):
    print(f"WARNING: NEGATIVE_LABEL={NEGATIVE_LABEL!r} was not found in the test labels.")
    print("Update NEGATIVE_LABEL in the setup cell to the dataset's actual negative label.")
else:
    original_precision = precision_score(
        y_test, y_pred, pos_label=NEGATIVE_LABEL, average="binary", zero_division=0
    )
    original_recall = recall_score(
        y_test, y_pred, pos_label=NEGATIVE_LABEL, average="binary", zero_division=0
    )
    original_f1 = f1_score(
        y_test, y_pred, pos_label=NEGATIVE_LABEL, average="binary", zero_division=0
    )

    balanced_precision = precision_score(
        y_test, balanced_pred, pos_label=NEGATIVE_LABEL, average="binary", zero_division=0
    )
    balanced_recall = recall_score(
        y_test, balanced_pred, pos_label=NEGATIVE_LABEL, average="binary", zero_division=0
    )
    balanced_f1 = f1_score(
        y_test, balanced_pred, pos_label=NEGATIVE_LABEL, average="binary", zero_division=0
    )

    comparison = pd.DataFrame({
        "Metric (negative class)": ["Precision", "Recall", "F1-score"],
        "Original model": [original_precision, original_recall, original_f1],
        "Class-balanced model": [balanced_precision, balanced_recall, balanced_f1]
    })
    display(comparison.round(4))

**How to compare:** Class balancing often improves negative-class recall by making the model pay more attention to the minority class, but it can lower precision by flagging more non-negative reviews as negative. Compare all three measured scores and choose based on the business goal; do not assume balancing is better without checking the held-out test results.

## Activity 5 — Flag unseen customer feedback

In [ ]:
# Replace these examples with the exact unseen feedback snippets supplied by your instructor.
# No snippets were included in the prompt, so these are intentionally left empty.
new_feedback = [
    # "Example feedback snippet 1",
    # "Example feedback snippet 2",
]

if not new_feedback:
    print("Add the provided unseen feedback snippets to the new_feedback list, then rerun this cell.")
else:
    new_predictions = pipeline.predict(new_feedback)
    new_probabilities = pipeline.predict_proba(new_feedback)
    class_names = pipeline.named_steps["classifier"].classes_
    negative_index = list(class_names).index(NEGATIVE_LABEL) if NEGATIVE_LABEL in class_names else None

    results = pd.DataFrame({
        "text": new_feedback,
        "predicted_label": new_predictions
    })

    if negative_index is not None:
        results["negative_probability"] = new_probabilities[:, negative_index]
        results["flagged_as_negative"] = results["predicted_label"] == NEGATIVE_LABEL
    else:
        print("NEGATIVE_LABEL does not match a trained class. Update it in the setup cell.")
    display(results)

### Recommendation to the business

Use the model as a **triage aid**, not as an automatic final decision. Negative-class precision estimates how often reviews flagged as negative are truly negative, while negative-class recall estimates how many truly negative reviews the model successfully catches. If recall is low, some unhappy customers may be missed; if precision is low, the support team may spend time reviewing false alarms. Check the measured negative-class precision and recall above, review flagged cases periodically, and keep a human in the loop—especially for urgent or safety-related complaints.

## Optional: Save the misclassified examples and feedback predictions

In [ ]:
# Uncomment to save outputs as CSV files after running the relevant activities:
# misclassified.to_csv("misclassified_reviews.csv", index=False)
# If Activity 5 results were created, save them with:
# results.to_csv("new_feedback_predictions.csv", index=False)